# ArchaeoTrace: local contour-guide training

Use an existing Colab GPU runtime and your own Drive. Set the archive path and a verified implementation commit below, then **Run all**. The archive must freeze all 144 rights-checked, explicitly human-approved annotation revisions, including the 48 calibration/holdout reference targets and predeclared standard/hard difficulty flags, before training. Training/validation crops may remain unassessed. The train/validation loader never opens protected calibration/holdout image or label bytes. The notebook does not approve labels, buy GPU access, or read calibration/holdout samples for training.

The three seeds are 3407, 3408 and 3409. Each run uses ResNet18 ImageNet initialization, a small U-Net, masked BCE/Dice plus 0.2 soft-clDice, AdamW 1e-4, batch 2 with 8 accumulation steps, at most 60 epochs and patience 8. Checkpoints on Drive retain complete optimizer/RNG/sampler progress at optimizer boundaries.

A single reviewer supplies pilot evidence only. These runs and the exported guide do not establish research accuracy or publication ranking eligibility.


In [ ]:
from pathlib import Path

REPOSITORY = "https://github.com/lzpxilfe/AI-Vectorizer-for-Archaeology.git"
CODE_COMMIT = ""  # Set to the verified 40-character implementation commit.
DATASET_ZIP = "/content/drive/MyDrive/ArchaeoTrace/approved-dataset.zip"
MANIFEST_RELATIVE_PATH = "manifest.json"
OUTPUT_DIRECTORY = "/content/drive/MyDrive/ArchaeoTrace/learned-runs"
SEEDS = (3407, 3408, 3409)
RESUME_EXISTING = True
OOM_RETRY = False  # Deliberate retry: resume latest.pt at batch 1 / accumulation 16.
MODEL_LICENSE = "GPL-2.0-only"


Connect an existing GPU and mount your own Drive. The mount dialog is handled by Colab; no credentials are stored in the dataset or model bundle.


In [ ]:
import re, subprocess, sys
from google.colab import drive

if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Set CODE_COMMIT to the verified full implementation commit.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
drive.mount("/content/drive")


Fetch exactly the selected code and install the separate learning runtime. ImageNet encoder weights are obtained only later through explicit training initialization.


In [ ]:
repository_path = Path("/content") / ("archaeotrace-code-" + CODE_COMMIT[:12])
if not repository_path.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY, str(repository_path)], check=True)
    subprocess.run(["git", "fetch", "origin", CODE_COMMIT], cwd=repository_path, check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_COMMIT], cwd=repository_path, check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repository_path, text=True).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError("Existing checkout does not match the selected commit.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repository_path / "requirements-learning.txt")], check=True)
sys.path.insert(0, str(repository_path))
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs an existing CUDA GPU runtime.")
print("Verified code:", actual_commit, "GPU:", torch.cuda.get_device_name(0))


Copy your approved ZIP from Drive into a fresh local directory. Archive paths and symbolic links are checked before extraction. Training reads the local files; model outputs and resumable checkpoints remain on your Drive.


In [ ]:
import hashlib, os, shutil, stat, tempfile, zipfile
from pathlib import PurePosixPath

archive = Path(DATASET_ZIP)
if not archive.is_file():
    raise FileNotFoundError(archive)
archive_hasher = hashlib.sha256()
with archive.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        archive_hasher.update(block)
archive_hash = archive_hasher.hexdigest()
data_path = Path("/content") / ("archaeotrace-data-" + archive_hash[:16])
if not data_path.exists():
    stage = Path(tempfile.mkdtemp(prefix=".archaeotrace-data-", dir="/content"))
    try:
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                relative = PurePosixPath(member.filename)
                mode = member.external_attr >> 16
                if relative.is_absolute() or ".." in relative.parts or "\\" in member.filename or stat.S_ISLNK(mode):
                    raise ValueError("Unsafe archive member: " + member.filename)
                target = stage.joinpath(*relative.parts)
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with zipped.open(member) as source, target.open("xb") as destination:
                        shutil.copyfileobj(source, destination)
        (stage / ".archive-sha256").write_text(archive_hash)
        os.rename(stage, data_path)
    finally:
        if stage.exists():
            shutil.rmtree(stage)
if (data_path / ".archive-sha256").read_text() != archive_hash:
    raise RuntimeError("Local extraction does not match the selected archive.")
manifest_path = (data_path / MANIFEST_RELATIVE_PATH).resolve()
if not manifest_path.is_relative_to(data_path.resolve()):
    raise ValueError("Manifest must remain inside the extracted dataset.")
from learning.dataset import ContourDataset, load_dataset_manifest
# Global anti-leak/freeze checks use only declared manifest metadata.
frozen = load_dataset_manifest(manifest_path, verify_files=False)
declared = frozen.payload["samples"]
if len(declared) != 144:
    raise ValueError("This pilot notebook requires the complete frozen 144-crop cohort.")
for sample in declared:
    annotation = sample.get("annotation", {})
    if annotation.get("status") != "approved":
        raise ValueError("Freeze all 144 annotation approvals before training.")
    if sample["split"] in ("calibration", "holdout") and (
        sample.get("difficulty") not in ("standard", "hard")
        or not sample.get("prompt") or not sample.get("reference_sha256")
    ):
        raise ValueError("Freeze all 48 evaluation targets and standard/hard difficulty flags before training.")
    if any(not re.fullmatch(r"[0-9a-f]{64}", annotation.get(key, "")) for key in ("sha256", "revision")):
        raise ValueError("Each frozen annotation needs its exact revision and file digest.")
training = ContourDataset(manifest_path, "train")
validation = ContourDataset(manifest_path, "validation")
print("Dataset SHA-256:", training.manifest_sha256, "approved train/validation:", len(training), len(validation))
run_root = Path(OUTPUT_DIRECTORY)
run_root.mkdir(parents=True, exist_ok=True)


Train or resume all three seeds. A CUDA OOM stops with the last valid checkpoint intact. Set `OOM_RETRY=True` and run this cell again to deliberately resume at batch 1 / accumulation 16; the effective batch stays 16. CPU execution is reserved for explicit synthetic diagnostics, not selected silently.


In [ ]:
from learning.train import TrainConfig, load_checkpoint, train_run

results = {}
for seed in SEEDS:
    run_directory = run_root / ("seed-" + str(seed))
    latest = run_directory / "latest.pt"
    resume = latest if RESUME_EXISTING and latest.exists() else None
    config = TrainConfig(seed=seed)
    if resume:
        config = TrainConfig(**load_checkpoint(resume)["config"])
        if config.seed != seed:
            raise ValueError("Resume checkpoint seed does not match its run directory.")
    results[seed] = train_run(
        manifest_path, run_directory, config=config, resume=resume,
        initialize_pretrained=resume is None, device="cuda", retry_oom=OOM_RETRY and resume is not None,
    )
    print(seed, results[seed]["stopped_reason"], results[seed]["progress"]["epoch"])


Select exactly one candidate by the lowest validation loss, breaking ties by ascending seed. Freeze that choice in `selected-model.json` before any calibration or holdout access. Export only the selected seed's best checkpoint. CPU PyTorch/ONNX probability parity must be at most 1e-4. Fixed 512 tiles with 384 cores and 64 halos are pan-consistent and do not promise whole-image equivalence.


In [ ]:
import json, math
from learning.export import export_checkpoint, validate_validation_selection
from learning.train import canonical_sha256
from ai_vectorizer.core.learned_model import inspect_bundle

# Candidate ranking uses only the frozen train/validation protocol.
candidates = []
for seed in SEEDS:
    checkpoint = run_root / ("seed-" + str(seed)) / "best.pt"
    record = load_checkpoint(checkpoint)
    loss = record["progress"]["best_validation"]
    if loss is None or not math.isfinite(loss):
        raise ValueError("Each seed must have a finite validation-selected checkpoint.")
    if record["dataset_manifest_sha256"] != training.manifest_sha256 or record["code"]["commit"] != CODE_COMMIT:
        raise ValueError("Candidate dataset/code identity differs from the pinned run.")
    if record["synthetic_diagnostic"]:
        raise ValueError("Synthetic diagnostics cannot be selected as real pilot candidates.")
    candidates.append({
        "seed": seed, "validation_loss": loss, "checkpoint": str(checkpoint),
        "checkpoint_sha256": record["checkpoint_file_sha256"],
        "training_config_sha256": record["config_sha256"],
        "training_source_sha256": record["code"]["source_sha256"],
    })
    del record
selected = min(candidates, key=lambda value: (value["validation_loss"], value["seed"]))
selection = {
    "schema_version": "archaeotrace-validation-selection/1",
    "selection_uses": "validation_only", "tie_break": "ascending_seed",
    "dataset_manifest_sha256": training.manifest_sha256, "code_commit": CODE_COMMIT,
    "selected": selected, "candidates": candidates,
    "calibration_accessed": False, "holdout_accessed": False,
    "publication_ranking_eligible": False,
}
selection_path = run_root / "selected-model.json"
if selection_path.exists():
    if json.loads(selection_path.read_text()) != selection:
        raise RuntimeError("Candidate selection is already frozen; use a new experiment directory.")
else:
    descriptor, temporary = tempfile.mkstemp(prefix=".selection-", dir=run_root)
    try:
        with os.fdopen(descriptor, "w") as stream:
            json.dump(selection, stream, indent=2, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        # Drive mounts support rename, while hard links may be unavailable.
        with selection_path.open("x"):
            pass
        os.replace(temporary, selection_path)
    finally:
        Path(temporary).unlink(missing_ok=True)
checkpoint = Path(selected["checkpoint"])
export_directory = checkpoint.parent / ("bundle-" + selected["checkpoint_sha256"][:12])
if export_directory.exists():
    existing = inspect_bundle(export_directory)
    if not existing.ready or existing.manifest["provenance"]["training_checkpoint_sha256"] != selected["checkpoint_sha256"]:
        raise RuntimeError("Existing bundle is unavailable or belongs to another checkpoint.")
    exported = existing.manifest
else:
    exported = export_checkpoint(checkpoint, export_directory, manifest_path,
        license_spdx=MODEL_LICENSE, selection_file=selection_path)
if exported["provenance"]["training_checkpoint_sha256"] != selected["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint changed after validation selection.")
provenance = exported["provenance"]
validate_validation_selection(selection, provenance)
if (provenance.get("validation_selection") != selection
    or provenance.get("validation_selection_sha256") != canonical_sha256(selection)
    or provenance.get("validation_selection_file_sha256") != hashlib.sha256(selection_path.read_bytes()).hexdigest()):
    raise RuntimeError("Exported bundle does not bind the frozen validation selection.")
print("Validation-only selected seed:", selected["seed"], "bundle:", export_directory)
print("Model SHA-256:", exported["model"]["sha256"], "frozen selection:", selection_path)
